<a href="https://colab.research.google.com/github/faheem1867/zero_to_hero/blob/main/Language_Identification_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip -q install tensorflow scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import zipfile

# Extract the dataset if a ZIP file was uploaded
for file in uploaded.keys():
    if file.endswith(".zip"):
        with zipfile.ZipFile(file, "r") as z:
            z.extractall("/content/language_data")

# Find CSV files
csv_files = glob.glob("/content/*.csv")
csv_files += glob.glob("/content/language_data/**/*.csv",
                       recursive=True)

if not csv_files:
    raise FileNotFoundError("Please upload a CSV or ZIP dataset.")

df = pd.read_csv(csv_files[0])

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

In [ ]:
# Normalize column names
df.columns = df.columns.str.strip().str.lower()

text_candidates = ["text", "sentence", "content"]
label_candidates = ["language", "lang", "label"]

text_col = next(
    (c for c in text_candidates if c in df.columns), None
)
label_col = next(
    (c for c in label_candidates if c in df.columns), None
)

if text_col is None or label_col is None:
    raise ValueError(
        f"Check dataset columns: {df.columns.tolist()}"
    )

# Keep only the required columns
df = df[[text_col, label_col]].copy()
df.columns = ["text", "language"]

# Handle missing values
df = df.dropna(subset=["text", "language"])

# Convert to strings and remove empty text
df["text"] = df["text"].astype(str).str.strip()
df["language"] = df["language"].astype(str).str.strip()

df = df[
    (df["text"] != "") &
    (df["language"] != "")
]

# Remove duplicate records
df = df.drop_duplicates()

# Select a manageable, balanced subset
min_samples = 100
counts = df["language"].value_counts()
valid_languages = counts[counts >= min_samples].index

df = df[df["language"].isin(valid_languages)]

# Up to 500 samples per language
df = (
    df.groupby("language", group_keys=False)
      .sample(n=500, replace=False, random_state=42)
      if (df["language"].value_counts() >= 500).all()
      else df.groupby("language", group_keys=False)
              .apply(lambda x: x.sample(
                  min(len(x), 500), random_state=42
              ))
              .reset_index(drop=True)
)

print("Cleaned dataset shape:", df.shape)
print("Number of languages:", df["language"].nunique())
print(df["language"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 100

# Encode language labels
encoder = LabelEncoder()
y = encoder.fit_transform(df["language"])

# Split the data before fitting the tokenizer
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"].values,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Tokenize text using training data only
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_text),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Languages:", encoder.classes_)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

num_classes = len(encoder.classes_)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64),
    LSTM(64),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("Accuracy :", round(
    accuracy_score(y_test, y_pred), 4
))
print("Precision:", round(
    precision_score(y_test, y_pred, average="weighted",
                    zero_division=0), 4
))
print("Recall   :", round(
    recall_score(y_test, y_pred, average="weighted",
                 zero_division=0), 4
))
print("F1-score :", round(
    f1_score(y_test, y_pred, average="weighted",
             zero_division=0), 4
))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(num_classes),
    target_names=encoder.classes_,
    zero_division=0
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test, y_pred,
    labels=np.arange(num_classes)
)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True if num_classes <= 15 else False,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.xlabel("Predicted Language")
plt.ylabel("Actual Language")
plt.title("Confusion Matrix - LSTM Language Identification")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
def predict_language(text):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    probabilities = model.predict(padded, verbose=0)[0]
    predicted_id = np.argmax(probabilities)

    language = encoder.inverse_transform([predicted_id])[0]
    confidence = probabilities[predicted_id]

    return language, confidence


unseen_samples = [
    "Hello, how are you?",
    "Hola, ¿cómo estás?",
    "Bonjour, comment allez-vous ?",
    "Guten Tag, wie geht es Ihnen?",
    "Ciao, come stai?"
]

for text in unseen_samples:
    language, confidence = predict_language(text)
    print("Text:", text)
    print("Predicted language:", language)
    print("Confidence:", f"{confidence:.2%}")
    print("-" * 40)

In [ ]:
model.save("language_identification_lstm.keras")

print("Model saved successfully!")

In [ ]:
print("Test Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nLanguage distribution:")
print(df["language"].value_counts())

print("\nPredicted language distribution:")
print(pd.Series(
    encoder.inverse_transform(y_pred)
).value_counts())

In [ ]:
print("Languages:", encoder.classes_)

print("\nSample training texts:")
for i in range(min(5, len(X_train_text))):
    print(X_train_text[i], "->",
          encoder.inverse_transform([y_train[i]])[0])

print("\nPrediction probabilities:")
print(np.round(y_prob[:5], 3))

In [ ]:
# Check random samples from each language
for lang in ["English", "French", "Chinese", "Tamil"]:
    print("\nLANGUAGE:", lang)
    samples = df[df["language"] == lang]["text"].head(5)

    for sample in samples:
        print("-", sample)

In [ ]:
print("Total samples:", len(df))
print("Unique languages:", df["language"].nunique())
print("Missing values:\n", df.isnull().sum())
print("\nSamples per language:\n",
      df["language"].value_counts())

In [ ]:
for lang in ["English", "French", "Chinese", "Tamil"]:
    print("\n---", lang, "---")

    samples = df[df["language"] == lang]["text"].head(3)

    for text in samples:
        print(text)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

MAX_WORDS = 30000
MAX_LEN = 120

# Fit tokenizer on training text
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_new = pad_sequences(
    tokenizer.texts_to_sequences(X_train_text),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_new = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

model = Sequential([
    Embedding(MAX_WORDS, 128),
    Bidirectional(LSTM(64)),
    Dropout(0.4),
    Dense(len(encoder.classes_), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_new,
    y_train,
    validation_split=0.1,
    epochs=12,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

y_prob = model.predict(X_test_new, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("Test Accuracy:",
      round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_,
    zero_division=0
))